# ⚗️ The Alchemist's Lab — Potion Brawl

*A trustworthy artifact in a booby-trapped lab. Everything below actually runs.*

Three enchanted potions bounce around the shop floor. When two collide, the stronger reagent transmutes the weaker into its own kind — rock‑paper‑scissors with bottles:

> 🔴 **Emberfury** *scorches* 🟢 **Verdant** *drinks* 🔵 **Tide** *douses* 🔴 **Emberfury** …

But the real spell we're demonstrating is the **virtual environment**. This notebook leans on a whole shelf of reagents (`numpy`, `scipy`, `matplotlib`, `plotly`, `networkx`, `imageio`, `rich`, `pyfiglet`, …). Pin them in `requirements.txt`, rebuild the venv anywhere, and the brew is identical — you can even `cp` the lab to a new machine and **pick up your work exactly where you left off.**

Run me with the **“Potion Brawl (venv)”** kernel: *Kernel → Restart & Run All*.

## 1. Which lab are we standing in?

Before anything else: *prove* we're inside the sealed venv, not the cluster's system Python. `sys.executable` should point inside `.venv/`, and every reagent on the recipe scroll should report a version.

In [ ]:
import sys
import importlib.metadata as im
import pandas as pd

print("Python                :", sys.version.split()[0])
print("Which python (sys.executable):", sys.executable)
print("Sealed lab (sys.prefix)      :", sys.prefix)
print("Is this a venv?              :", sys.prefix != sys.base_prefix)

reagents = ["numpy", "scipy", "pandas", "matplotlib", "plotly",
            "networkx", "imageio", "pillow", "rich", "pyfiglet", "tqdm"]
rows = []
for name in reagents:
    try:
        rows.append((name, im.version(name)))
    except im.PackageNotFoundError:
        rows.append((name, "❌ MISSING"))
pd.DataFrame(rows, columns=["reagent", "version"]).set_index("reagent")

## 2. The Law of the Brawl

The three potions form a cycle — each one transmutes exactly one other. `networkx` draws the law.

In [ ]:
%matplotlib inline
from potion_brawl import Brawl, POTIONS, plot_cycle, plot_populations, plotly_scene

plot_cycle();

## 3. Brew a batch

Toss 120 bottles (40 of each potion) into a 3‑D cauldron and let them brawl for 400 ticks. The stacked‑area chart shows the fortunes of war.

In [ ]:
brawl = Brawl(n_per_type=40, dim=3, size=8.0, seed=7)
brawl.run(400, capture_every=4)

print("final tally:", {p.element: int(c) for p, c in zip(POTIONS, brawl.counts())})
plot_populations(brawl);

## 4. The 3‑D shop floor

Here's the brawl itself — an interactive `plotly` scene. Hit **▶ Brew** and drag to orbit the cauldron.

In [ ]:
plotly_scene(brawl).show()

## 5. Pick up exactly where you left off

This is the whole point. We run one brew straight through, and a second brew that we **pause, save to a lab journal, walk away from, and reopen** — as if in a brand‑new lab. The journal stores positions, velocities, the tick count, *and* the RNG state, so the resumed brew is **bit‑for‑bit identical** to the uninterrupted one.

In [ ]:
import os
os.makedirs("output", exist_ok=True)
journal = "output/demo_journal.pkl"

# One uninterrupted brew...
full = Brawl(seed=123); full.run(80, progress=False)

# ...versus a brew we pause, save, 'close the lab', and resume 'elsewhere'.
part = Brawl(seed=123); part.run(40, progress=False)
part.save(journal)
del part                          # walk away from the bench
resumed = Brawl.load(journal)     # reopen the journal in a 'new lab'
resumed.run(40, progress=False)

print("uninterrupted final counts:", full.counts().tolist())
print("paused + resumed  counts  :", resumed.counts().tolist())
print("bit-identical histories?  ->", full.history == resumed.history)

## Why this matters — the moral of the brew ⚗️

- **The venv is a sealed laboratory.** Cell 1 proved `sys.executable` lives inside `.venv/`. Reagents summoned here never leak out onto the cluster, and the cluster's Python never contaminates the brew.
- **`requirements.txt` is the recipe scroll.** It reconstitutes the *exact* shelf of reagents in any lab. That's why the version table in cell 1 is reproducible for everyone who builds from it.
- **You rebuild a venv, you don't copy one.** Copying a `.venv/` folder fails — see this lab's `former_apprentice/.venv/`, whose Python symlink points at an account that isn't yours. The scroll is portable; the sealed lab is not.
- **The lab journal lets you resume anywhere.** `cp` the code + `requirements.txt` (+ `output/lab_journal.pkl`) to a new directory or machine, rebuild the reagents, and the brew continues mid‑pour.

```bash
cp -r potion_brawl ~/a_new_lab
cd ~/a_new_lab
/usr/bin/python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt      # the scroll rebuilds the shelf
python potion_brawl.py               # 📜 resumes the brew, mid-brawl
```

*Reproducibility you can see.* ⚗️